In [ ]:
import os

import pandas as pd
import numpy as np
import re
import matplotlib as plt

import uk_postcodes_parsing as ukp

import random

from datetime import date, datetime, timedelta
from dateutil.relativedelta import relativedelta

data_path = os.path.join('..', 'data/')
raw_data_path = os.path.join(data_path, 'raw_data')
processed_data_path = os.path.join(data_path, 'processed_data')

## Import Inclusion Patients

In [ ]:
inclusion_patients_df = pd.read_csv(os.path.join(data_path, "chronic_kidney_disease_refined_inclusion_patients.csv"))

try:
    inclusion_patients_df.rename(columns={"measureDate": "inclusion_date"}, inplace=True)
    inclusion_patients_df.rename(columns={"deathOfDate": "dateOfDeath"}, inplace=True)
    inclusion_patients_df.drop(columns=['addl_pat_flg', 'raceCode'], inplace=True)
except:
    pass

inclusion_patients_df['dateOfBirth'] = pd.to_datetime(inclusion_patients_df['dateOfBirth'], format='mixed').dt.date
inclusion_patients_df['dateOfDeath'] = pd.to_datetime(inclusion_patients_df['dateOfDeath'], format='mixed').dt.date
inclusion_patients_df['inclusion_date'] = pd.to_datetime(inclusion_patients_df['inclusion_date'], format='mixed').dt.date
inclusion_patients_df['endpoint_date'] = inclusion_patients_df['endpoint_date'].apply(lambda x: '2025-12-31' if pd.isna(x) else x)
inclusion_patients_df['endpoint_date'] = pd.to_datetime(inclusion_patients_df['endpoint_date']).dt.date

inclusion_patients_df['postcode'] = inclusion_patients_df['postcode'].astype(str)

inclusion_patients_df.head()

### Remove Patients with Multiple NHS Numbers

In [ ]:
identifier3_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[1].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower(): 
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            identifier3_dict[nhs_num]=row[0]

In [ ]:
opt_in_check_df = pd.DataFrame().from_dict(identifier3_dict, orient='index').reset_index()

del identifier3_dict

opt_in_check_df.columns=['patient_identifier3', 'master_person_id']

In [ ]:
df = opt_in_check_df['master_person_id'].value_counts().reset_index()

multiple_patients = list(set(df[df['count']>1]['master_person_id'].to_list()))

del df

opt_in_check_df = opt_in_check_df[~opt_in_check_df['master_person_id'].isin(multiple_patients)]

### Remove Opt-Out Patients

In [ ]:
opt_in_patients_df = pd.read_csv(os.path.join(raw_data_path, "nhs_opt_in_patients.csv"))

opt_in_patients_df.columns = ['patient_identifer3']

opt_in_patients_df['patient_identifer3'] = opt_in_patients_df['patient_identifer3']

opt_in_nhs_num_list = list(set(opt_in_patients_df['patient_identifer3'].to_list()))

del opt_in_patients_df

print(f'Number of Opt-In Patients: {len(opt_in_nhs_num_list):,}')

In [ ]:
opt_in_person_list = opt_in_check_df[opt_in_check_df['patient_identifier3'].isin(opt_in_nhs_num_list)]['master_person_id'].to_list()

inclusion_patients_refined_df = inclusion_patients_df[inclusion_patients_df['master_person_id'].isin(opt_in_person_list)]

del opt_in_nhs_num_list, opt_in_person_list, opt_in_check_df

### Expand Postcode Details

In [ ]:
postcode_df = pd.read_csv(os.path.join(raw_data_path, 'elasticsearch_search_hits/postcode_extract.csv'))

inclusion_patients_df = inclusion_patients_df.merge(postcode_df, how='left', on='master_person_id')

inclusion_patients_df['postcode'] = inclusion_patients_df['postcode'].apply(lambda x: x if x!='nan' else np.NaN)
inclusion_patients_df['postcode'] = inclusion_patients_df['postcode'].combine_first(inclusion_patients_df['patient_AddressPostalCode'])

inclusion_patients_df['postcode'] = inclusion_patients_df['postcode'].astype(str)

inclusion_patients_df = inclusion_patients_df.drop(columns=['patient_AddressPostalCode'])

inclusion_patients_df.head()

#### Find Postcode District and Region

In [ ]:
def postcode_district(postcode):
    if ukp.lookup_postcode(postcode) == None:
        if postcode[:2] == 'ZZ':
            return 'Unknown'
        elif postcode[:2] =='GX':
            return 'Gibraltar'
        elif postcode[:2] =='GY':
            return 'Channel Islands'
        else:
            return np.NaN
    elif ukp.lookup_postcode(postcode).district == None:
        return ukp.lookup_postcode(postcode).country
    else:
        return ukp.lookup_postcode(postcode).district

In [ ]:
def postcode_region(postcode):
    if ukp.lookup_postcode(postcode) == None:
        if postcode[:2] == 'ZZ':
            return 'Unknown'
        elif postcode[:2] =='GX':
            return 'Gibraltar'
        elif postcode[:2] =='GY':
            return 'Channel Islands'
        else:
            return np.NaN
    elif ukp.lookup_postcode(postcode).region == None:
        return 'Unkown'
    else:
        return ukp.lookup_postcode(postcode).region

In [ ]:
inclusion_patients_refined_df.insert(9, 'postcode_district', inclusion_patients_refined_df['postcode'].apply(lambda x: postcode_district(x)))
inclusion_patients_refined_df.insert(10, 'postcode_region', inclusion_patients_refined_df['postcode'].apply(lambda x: postcode_region(x)))

inclusion_patients_refined_df.head()

#### Populate Details for Non-Standardly Formatted Postcodes

In [ ]:
postcode_dict = {}
for code in inclusion_patients_refined_df[inclusion_patients_refined_df['postcode_district'].notna()]['postcode'].to_list():
    try:
        index = code.index(' ')
        postcode_dict[code[:index]] = code
    except:
        pass

In [ ]:
def postcode_second_pass(row):
    if str(row['postcode_district'])=='nan':
        postcode = row['postcode']
        try:
            index = postcode.index(' ')
        except:
            index = None
    
        if index:
            try:
                return ukp.lookup_postcode(postcode_dict[postcode[:index]]).district
            except:
                return np.NaN
        elif len(postcode)==3 or len(postcode)==4:
            try:
                return ukp.lookup_postcode(postcode_dict[postcode[:index]]).district
            except:
                return 'Unknown'
        else:
            return 'Unknown'
    else:
        return row['postcode_district']

In [ ]:
inclusion_patients_refined_df['postcode_district'] = inclusion_patients_refined_df.apply(lambda x: postcode_second_pass(x), axis=1)

In [ ]:
postcode_dict = {}
for code in inclusion_patients_refined_df[inclusion_patients_refined_df['postcode_region'].notna()]['postcode'].to_list():
    try:
        index = code.index(' ')
        postcode_dict[code[:index]] = code
    except:
        pass

In [ ]:
def region_second_pass(row):
    if str(row['postcode_region'])=='nan':
        postcode = row['postcode']
        try:
            index = postcode.index(' ')
        except:
            index = None
    
        if index:
            try:
                return ukp.lookup_postcode(postcode_dict[postcode[:index]]).region
            except:
                return np.NaN
        elif len(postcode)==3 or len(postcode)==4:
            try:
                return ukp.lookup_postcode(postcode_dict[postcode[:index]]).region
            except:
                return 'Unknown'
        else:
            return 'Unknown'
    else:
        return row['postcode_region']

In [ ]:
inclusion_patients_refined_df['postcode_region'] = inclusion_patients_refined_df.apply(lambda x: region_second_pass(x), axis=1)

#### Export Postcode District & Region Breakdowns

In [ ]:
postcode_district_breakdown_df = inclusion_patients_refined_df[['postcode_district', 'postcode_region']].value_counts(dropna=False).reset_index()

In [ ]:
postcode_district_breakdown_df.to_csv(os.path.join(data_path, 'postcode_district_breakdown.csv'), index=False)

#### Import and Filter for Relevant Districts

In [ ]:
relevant_postcode_districts_df = pd.read_csv(os.path.join(data_path, 'relevant_postcode_districts.csv'))

relevant_postcode_districts_df = relevant_postcode_districts_df[relevant_postcode_districts_df['include_district']==1]

relevant_districts_list = relevant_postcode_districts_df['postcode_district'].to_list()
gstt_dialysis_districts_list = relevant_postcode_districts_df[relevant_postcode_districts_df['gstt_dialysis_districts']==1]['postcode_district'].to_list()
gfr_slope_districts_list = relevant_postcode_districts_df[relevant_postcode_districts_df['gfr_slope_districts']==1]['postcode_district'].to_list()

del relevant_postcode_districts_df

In [ ]:
inclusion_patients_refined_df = inclusion_patients_refined_df[inclusion_patients_refined_df['postcode_district'].isin(relevant_districts_list)].reset_index(drop=True)

inclusion_patients_refined_df.insert(11, 'gstt_dialysis_districts', inclusion_patients_refined_df['postcode_district'].apply(lambda x: 1 if x in gstt_dialysis_districts_list else 0))
inclusion_patients_refined_df.insert(12, 'gfr_slope_districts', inclusion_patients_refined_df['postcode_district'].apply(lambda x: 1 if x in gfr_slope_districts_list else 0))

del relevant_districts_list, gstt_dialysis_districts_list, gfr_slope_districts_list

inclusion_patients_refined_df.head()

In [ ]:
print(f'Total Number of Relevant Patients: {inclusion_patients_refined_df.shape[0]:,}')
print(f'Number of GSTT Dialys Patients: {inclusion_patients_refined_df[inclusion_patients_refined_df['gstt_dialysis_districts']==1].shape[0]:,}')
print(f'Number of GFR Slope Patients: {inclusion_patients_refined_df[inclusion_patients_refined_df['gfr_slope_districts']==1].shape[0]:,}')

### Standardise Race

In [ ]:
def race_standardisation(race):
    race = str(race)
    if race == 'nan':
        return 'Unknown'
    elif 'mixed:' in race.lower() or 'mixed -' in race.lower():
        if 'white' in race.lower():
            if 'asian' in race.lower():
                return 'Mixed - White & Asian'
            elif 'caribbean' in race.lower():
                return 'Mixed - White & Black Caribbean'
            elif 'black british' in race.lower():
                return 'Mixed - Black British & White'
            elif 'african' in race.lower() or 'black' in race.lower():
                return 'Mixed - White & Black African'
            elif 'chinese' in race.lower():
                return 'Mixed - Chinese & White'
        elif 'asian' in race.lower():
            if 'chinese' in race.lower():
                return 'Mixed - Asian & Chinese'
            elif 'black' in race.lower():
                return 'Mixed - Black & Asian'
        else:
            return 'Mixed - Other'
    elif 'white:' in race.lower() or 'white -' in race.lower() or race.lower() == 'white':
        if 'british' in race.lower():
            return 'White - British'
        elif 'northern' in race.lower():
            return 'White - Northern Irish'
        elif 'english' in race.lower():
            return 'White - English'
        elif 'irish' in race.lower():
            return 'White - Irish'
        elif 'scottish' in race.lower():
            return 'White - Scottish'
        elif 'welsh' in race.lower():
            return 'White - Welsh'
        elif 'italian' in race.lower():
            return 'White - Italian'
        elif 'greek' in race.lower() and 'cypriot' in race.lower():
            return 'White - Greek Cypriot'
        elif 'turk' in race.lower() and 'cypriot' in race.lower():
            return 'White - Turkish Cypriot'
        elif 'cypriot' in race.lower():
            return 'White - Cypriot (Unspecified)'
        elif 'greek' in race.lower():
            return 'White - Greek'
        elif 'cornish' in race.lower():
            return 'White - Cornish'
        elif 'portuguese' in race.lower():
            return 'White - Portuguese'
        elif 'turkish' in race.lower():
            return 'White - Turkish'
        elif 'gypsy' in race.lower() or 'romany' in race.lower():
            return 'White - Gypsy/Romany'
        elif 'bosnian' in race.lower():
            return 'White - Bosnian'
        elif 'ussr' in race.lower():
            return 'White - Ex-USSR'
        elif 'yugoslav' in race.lower():
            return 'White - Yugoslavian'
        elif 'kosov' in race.lower():
            return 'White - Kosovan'
        elif 'polish' in race.lower():
            return 'White - Polish'
        elif 'albania' in race.lower():
            return 'White - Albanian'
        elif 'serbian' in race.lower():
            return 'White - Serbian'
        elif 'croatian' in race.lower():
            return 'White - Croatian'
        elif 'other white' in race.lower():
            return 'White - Other White Background'
        elif 'other europ' in race.lower():
            return 'White - Other European Background'
        else:
            return 'White - Unspecified'
    elif 'black:' in race.lower() or 'black -' in race.lower() or 'black or' in race.lower():
        if 'black british' in race.lower():
            if 'african' in race.lower():
                return 'Black or Black British - African'
            elif '- british' in race.lower():
                return 'Black or Black British - British'
            elif 'caribbean' in race.lower():
                return 'Black or Black British - Caribbean'
            elif 'somali' in race.lower():
                return 'Black or Black British - Somali'
            else:
                return 'Black or Black British - Other'
        else:
            if 'british' in race.lower():
                return 'Black - Black British'
            elif 'eritrean' in race.lower():
                return 'Black - Eritrean'
            elif 'ethiopian' in race.lower():
                return 'Black - Ethiopian'
            elif 'ghanaian' in race.lower():
                return 'Black - Ghanaian'
            elif 'mixed' in race.lower():
                return 'Black - Mixed'
            elif 'somali' in race.lower():
                return 'Black - Somali'
            elif 'sudanese' in race.lower():
                return 'Black - Sudanese'
            elif 'ugandan' in race.lower():
                return 'Black - Ugandan'
            else:
                return 'Black - Other'
    elif 'asian:' in race.lower() or 'asian -' in race.lower() or 'asian or' in race.lower() or 'asian  or' in race.lower():
        if 'british -' in race.lower():
            if 'arab' in race.lower():
                return 'Asian or Asian British - Arab'
            elif 'bangladeshi' in race.lower():
                return 'Asian or Asian British - Bangladeshi'
            elif 'british' in race.lower():
                return 'Asian or Asian British - British'
            elif 'caribbean' in race.lower():
                return 'Asian or Asian British - Caribbean'
            elif 'chinese' in race.lower():
                return 'Asian or Asian British - Chinese'
            elif 'east african' in race.lower():
                return 'Asian or Asian British - East African'
            elif 'filpino' in race.lower():
                return 'Asian or Asian British - Filpino'
            elif 'indian' in race.lower():
                return 'Asian or Asian British - Indian'
            elif 'japanese' in race.lower():
                return 'Asian or Asian British - Japanese'
            elif 'malaysian' in race.lower():
                return 'Asian or Asian British - Malaysian'
            elif 'pakistani' in race.lower():
                return 'Asian or Asian British - Pakistani'
            elif 'punhabi' in race.lower():
                return 'Asian or Asian British - Punhabi'
            elif 'sinhalese' in race.lower():
                return 'Asian or Asian British - Sinhalese'
            elif 'sri lankan' in race.lower():
                return 'Asian or Asian British - Sri Lankan'
            elif 'tamil' in race.lower():
                return 'Asian or Asian British - Tamil'
            elif 'vietnamese' in race.lower():
                return 'Asian or Asian British - Vietnamese'
            else:
                return 'Asian or Asian British - Other'
        else:
            if 'east african asian' in race.lower():
                return 'Asian - East African Asian'
            elif 'british asian' in race.lower():
                return 'Asian - British Asian'
            elif 'caribbean asian' in race.lower():
                return 'Asian - Caribbean Asian'
            elif 'mixed asian' in race.lower():
                return 'Asian - Mixed Asian'
            elif 'punjabi' in race.lower():
                return 'Asian - Punjabi'
            elif 'sri lankan' in race.lower():
                return 'Asian - Sri Lankan'
            elif 'tamil' in race.lower():
                return 'Asian - Tamil'
            elif 'sihnalese' in race.lower():
                return 'Asian - Sihnalese'
            else:
                return 'Asian - Other'
    elif 'other' in race.lower():
        if 'arab' in race.lower():
            return 'Other - Arab'
        if 'chinese' in race.lower():
            return 'Other - Chinese'
        if 'columbian' in race.lower():
            return 'Other - Columbian'
        if 'ecuadorian' in race.lower():
            return 'Other - Ecuadorian'
        if 'filipino' in race.lower():
            return 'Other - Filipino'
        if 'iranain' in race.lower():
            return 'Other - Iranain'
        if 'iraqi' in race.lower():
            return 'Other - Iraqi'
        if 'japanese' in race.lower():
            return 'Other - Japanese'
        if 'latin american' in race.lower():
            return 'Other - Latin American'
        if 'malaysian' in race.lower():
            return 'Other - Malaysian'
        if 'middle eastern' in race.lower():
            return 'Other - Middle Eastern'
        if 'vietnamese' in race.lower():
            return 'Other - Vietnamese'
        else:
            return 'Other - Other'
    elif 'not' in race.lower() and ('stated' in race.lower() or 'given' in race.lower() or 'specified' in race.lower()):
        return 'Not Stated'
    else:
        return race

In [ ]:
inclusion_patients_refined_df.insert(8, 'raceDetailed', inclusion_patients_refined_df['race'].apply(lambda x: race_standardisation(x)))

In [ ]:
def race_refined(race):
    if 'White - ' in race:
        return 'White'
    elif 'Black -' in race or 'Black or' in race:
        return 'Black'
    elif 'Asian -' in race or 'Asian or' in race:
        return 'Asian'
    elif race == 'Unknown' or race == 'Not Stated':
        return 'Not Stated'
    else:
        return 'Other'

In [ ]:
inclusion_patients_refined_df['race'] = inclusion_patients_refined_df['raceDetailed'].apply(lambda x: race_refined(x))

### Remove Patients where Endpoint Data is before Inclusion Date

In [ ]:
inclusion_patients_refined_df = inclusion_patients_refined_df[inclusion_patients_refined_df['endpoint_date']>=inclusion_patients_refined_df['inclusion_date']]

### Save Opt-In Patients

In [ ]:
inclusion_patients_refined_df.to_csv(os.path.join(data_path, "opt_in_ckd_inclusion_patients.csv"), index=False)

## Create 6-Monthly Breakdowns

In [ ]:
inclusion_patients_refined_df = pd.read_csv(os.path.join(data_path, "opt_in_ckd_inclusion_patients.csv"))

inclusion_patients_refined_df['dateOfBirth'] = pd.to_datetime(inclusion_patients_refined_df['dateOfBirth'], format='mixed').dt.date
inclusion_patients_refined_df['dateOfDeath'] = pd.to_datetime(inclusion_patients_refined_df['dateOfDeath'], format='mixed').dt.date
inclusion_patients_refined_df['inclusion_date'] = pd.to_datetime(inclusion_patients_refined_df['inclusion_date'], format='mixed').dt.date
inclusion_patients_refined_df['endpoint_date'] = inclusion_patients_refined_df['endpoint_date'].apply(lambda x: '2025-12-31' if pd.isna(x) else x)
inclusion_patients_refined_df['endpoint_date'] = pd.to_datetime(inclusion_patients_refined_df['endpoint_date']).dt.date

In [ ]:
inc_dates_df = inclusion_patients_refined_df[['master_person_id', 'inclusion_date', 'endpoint_date']].drop_duplicates().reset_index(drop=True)

inc_dates_df['inclusion_date'] = pd.to_datetime(inc_dates_df['inclusion_date']).dt.date
inc_dates_df['endpoint_date'] = pd.to_datetime(inc_dates_df['endpoint_date']).dt.date

In [ ]:
df_list = []

for idx, row in inc_dates_df.iterrows():
    inc_date = row['inclusion_date']
    endpoint = row['endpoint_date']
    date = inc_date

    i = 1
    while date <= endpoint:
        row_list = []

        row_list.append(row['master_person_id'])
        row_list.append(inc_date)
        row_list.append(endpoint)
        row_list.append(i)
        row_list.append(date)

        date += relativedelta(months=12)

        if date >= endpoint:
            row_list.append(row['endpoint_date'])
        else:
            row_list.append(date)

        date += relativedelta(days=1)

        i += 1
        df_list.append(row_list)

In [ ]:
dates_breakdown_df = pd.DataFrame(df_list, columns=['master_person_id', 'inclusion_date', 'endpoint_date', 'grouping', 'start_date', 'end_date'])

dates_breakdown_df.head()

### Save Date Breakdown

In [ ]:
dates_breakdown_df.to_csv(os.path.join(data_path, "ckd_patients_datebreakdown.csv"), index=False)